In [ ]:
import os
import sys

import pandas as pd
import py7zr
import rarfile

from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.get_sha256_of_directories import (
    SEPARATOR,
    analyze_file,
    analyze_rar_file,
    analyze_sevenzip_file,
    analyze_tar_file,
    analyze_zip_file,
)
from pynxtools_em.examples.oasisb_utils import CSV_HEADER_FOR_HASH_FILE, get_project_id

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

## Compute hash values of each file of the projects

Using hash values practically resolves issues when different researchers name their files the same.<br>

In [ ]:
config: dict[str, str] = {
    "python_version": f"{sys.version.replace(' ', '_')}",
    "working_directory": f"{os.getcwd()}",
    "pynxtools_apm version": f"{get_pynxtools_em_version()}",
    "rarfile version": f"{rarfile.__version__}",
    "sevenzip version": f"{py7zr.__version__}",
    # "blake3-py version": f"{blake3.__version__}",
    # "blake3-py max_threads": f"{blake3.blake3.AUTO}",
    # "directory": f"src_directory,  # sys.argv[1],
}

spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
)

project_name_whitelist = sorted(
    (
        822,
        821,
        820,
        819,
        450,
        173,
        818,
        817,
        816,
        484,
        559,
        483,
        454,
        451,
        449,
        443,
        442,
        633,
        815,
        814,
        813,
        812,
        811,
        810,
        809,
        808,
        807,
        806,
        805,
        804,
        803,
        802,
        801,
        800,
        799,
        798,
        797,
        796,
        795,
        794,
        793,
        792,
        791,
        790,
        789,
        788,
        787,
        786,
        785,
        784,
        783,
        782,
        781,
        780,
        779,
        778,
        777,
        776,
        775,
        774,
        773,
        772,
        771,
        770,
        769,
        768,
        767,
        766,
        765,
        764,
        763,
        762,
        761,
        760,
        759,
        758,
        757,
        756,
        755,
        754,
        753,
        752,
        751,
        750,
        749,
        748,
        747,
        746,
        745,
        744,
        743,
        742,
        741,
        740,
        739,
        738,
        737,
        736,
        735,
        734,
        733,
        732,
        731,
        730,
        729,
        728,
        727,
        726,
        725,
        724,
        723,
        722,
        721,
        720,
        719,
        718,
        717,
        716,
        715,
        714,
        713,
        712,
        711,
        710,
        709,
        708,
        707,
        706,
        705,
        704,
        703,
        702,
        701,
        700,
        699,
        698,
        697,
        696,
        695,
        694,
        693,
        692,
        691,
        690,
        689,
        688,
        687,
        686,
        685,
        684,
        683,
        682,
        681,
        680,
        679,
        678,
        677,
        676,
        675,
        674,
        673,
        672,
        671,
        670,
        669,
        668,
        667,
        666,
        665,
        664,
        663,
        662,
        661,
        660,
        659,
        658,
        657,
        656,
        655,
        654,
        653,
        652,
        651,
        650,
        649,
        648,
        647,
        646,
        645,
        644,
        643,
        642,
        641,
        640,
        639,
        638,
        637,
        636,
        635,
        634,
        371,
        477,
        280,
    )
)
# project_name_blacklist = sorted(())

for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.parse in (1, 2):
        # if int(row.project_name) < 560:  # skip all before
        #     continue
        # if int(row.project_name) > 700:  # skip all after
        #     continue
        if int(row.project_name) not in project_name_whitelist:
            continue
        # if int(row.project_name) in project_name_blacklist:
        #     continue

        project_id = get_project_id(f"{row.project_name}")

        print(f"project{SEPARATOR}{project_id}{SEPARATOR}hashing...")
        sub_directory = f"{src_directory}{os.sep}{project_id}"
        results = []
        issues = []
        project_config = config
        prefix = f"{sub_directory}{os.sep}"
        project_config["directory"] = prefix
        for key, value in project_config.items():
            results.append(f"{key}{SEPARATOR}{value}")
            issues.append(f"{key}{SEPARATOR}{value}")
        del project_config, key, value
        results.append(CSV_HEADER_FOR_HASH_FILE)
        for root, dirs, files in os.walk(sub_directory):
            for file in files:
                fpath = f"{root}/{file}".replace(os.sep * 2, os.sep)
                # fname = os.path.basename(fpath)
                suffix = fpath.replace(config["directory"], "")

                if fpath.lower().endswith((".zip", ".eln")):
                    analyze_zip_file(fpath, results, issues, prefix)
                elif fpath.lower().endswith((".tar", ".tar.gz", ".tar.bz2", ".tar.xz")):
                    analyze_tar_file(fpath, results, issues, prefix)
                elif fpath.lower().endswith(".rar"):
                    analyze_rar_file(fpath, results, issues, prefix)
                elif fpath.lower().endswith(".7z"):
                    analyze_sevenzip_file(fpath, results, issues, prefix)
                else:
                    analyze_file(fpath, results, issues, prefix)
        # del root, dirs, files, file, fpath, suffix
        for name in ["root", "dirs", "files", "file", "fpath", "suffix"]:
            globals().pop(name, None)

        with open(
            f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
            "w",
            encoding="utf-8",
            errors="surrogateescape",
        ) as fp:
            fp.write("\n".join(results))
        del results
        with open(
            f"{src_directory}{os.sep}{project_id}.sha256.issues.csv",
            "w",
            encoding="utf-8",
            errors="surrogateescape",
        ) as fp:
            fp.write("\n".join(issues))
        if len(issues) > 6:
            print(issues)
        else:
            print(f"project{SEPARATOR}{project_id}{SEPARATOR}no issues")
        del issues, sub_directory, prefix

In [ ]:
# either empty directories or not existent
# 173, 559, 442, 443, 449, 450, 451, 452, 454